In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# BlindDetection-V2：阶段3机制与评分A/B

本包须经控制会话批准后由用户手动运行。复用阶段2前两对开发图，12张观测图；不是阶段4、正式检出率或FPR。先运行到首图计时单元，查看实测成本后手动继续剩余单元。

每个搜索最多200评分。两版独立选择top3，共享候选分数但不共享latent；每个候选保存原/优化全部17组LF/HF/weighted、m、排序差异、编码次数和计时。优化尚未获得真实等价或性能验证。


## 1. 已审阅代码与输入
固定原图、-11.7°、+18.3°联合(+0.75,-0.75)px，各正负两臂。细节见STAGE3_SCOPE.md。

In [ ]:
from pathlib import Path
import os,sys,json,subprocess,time
EXACT='def11a663fc2c8c16288bb4c4240eb8fe8813498'
REPO=Path('/content/ceg-wm-v2-stage3-def11a6')
INPUT=Path('/content/drive/MyDrive/CEG-WM/BlindDetection-V2/stage2-mechanism-v1')
OUTPUT=Path('/content/drive/MyDrive/CEG-WM/BlindDetection-V2/stage3-mechanism-v1')
if OUTPUT.exists():
    raise FileExistsError('保留已有stage3输出，不覆盖或自动重跑。')
required=[INPUT/f'v2-mechanism-{i:02d}__{arm}.png' for i in (0,1) for arm in ('content','clean')]
missing=[str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError(f'缺少既有阶段2开发图：{missing}')
if not REPO.exists():
    subprocess.run(['git','clone','--branch','BlindDetection-V2','--single-branch','https://github.com/RICHAAARC/CEG-WM.git',str(REPO)],check=True)
elif subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
    raise RuntimeError('保留已有代码改动。')
subprocess.run(['git','-C',str(REPO),'checkout','--detach',EXACT],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q',str(REPO)],check=True)
sys.path.insert(0,str(REPO))
print('Code:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

## 2. 密钥与环境
使用可运行现有内容模型的CUDA设备，不限定型号或显存。

In [ ]:
from google.colab import userdata
import torch
for name in ('HF_TOKEN','CEG_WM_ROOT_KEY'):
    try: value=userdata.get(name)
    except Exception: raise RuntimeError(f'请开启Colab Secret {name} 访问') from None
    if not value: raise RuntimeError(f'Empty secret: {name}')
    os.environ[name]=value
del value
print({'torch':torch.__version__,'cuda':torch.cuda.is_available(),'gpu':torch.cuda.get_device_name() if torch.cuda.is_available() else None})

## 3. 首图完整A/B计时

先只运行固定第0张图（unit00、无攻击、正样本）。输出实际图像耗时及剩余11图估算；搜索分叉会增加候选并集和实际用时。错误为0也不等于方法验证通过，应同时查看完整性与分支差异。


In [ ]:
from diagnostics.blind_detection_v2.stage3 import Session
begin=time.monotonic()
session=Session(INPUT,OUTPUT)
print('初始化秒数:',time.monotonic()-begin)
pilot=session.pilot()
print(json.dumps(pilot,indent=2))

## 4. 手动继续剩余11张

查看上一步实际成本后运行本单元。按固定顺序保留失败，不自动重跑，不改阈值。配对负样本与truth评分只用于诊断，truth不进入搜索。


In [ ]:
summary=session.remaining()
print(json.dumps(summary,indent=2))

## 5. 查看结果并交回控制会话
全量分支与候选在image-XX目录，summary含独立错误计数。先审阅真实A/B和机制结果，再决定下一步；不要将这两对开发图计入阶段4。

In [ ]:
rows=[json.loads(line) for line in (OUTPUT/'rows.jsonl').read_text().splitlines()] if (OUTPUT/'rows.jsonl').exists() else []
for row in rows:
    print({k:row.get(k) for k in ('index','condition','arm','error','ab_candidate_errors','top3_equal','ranking_equal','max_difference','max_abs_branch_difference','reference_boundary_changes','seconds')})
print({'planned_images':12,'written_images':len(rows),'unique_images':len({r['index'] for r in rows})})
if not (OUTPUT/'summary.json').exists(): print('未完成：没有终态summary；保留已有文件。')